In [169]:
from langgraph.graph import StateGraph,START,END
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from typing import TypedDict,Literal,Annotated
import operator
from pydantic import Field,BaseModel
from langchain_groq import ChatGroq




In [170]:
class post_create(TypedDict):
    topic : str =Field(description="Topic on which social platform post required")
    post : str
    evaluation : str = Literal["Approved","Not approved"]
    feedback : str
    iteration : int
    max_iteration : int


class postEvaluation(BaseModel):
    evaluation : Literal["Approved","Not approved"] = Field(description="Final evaluation")
    feedback : str = Field(description="Feedback on the post given mentioning merits and demerits in 100 words ")




In [171]:
llm = ChatGroq(model="llama-3.3-70b-versatile")
structured_model=llm.with_structured_output(postEvaluation)
#res=structured_model.invoke(post)



In [172]:
def generate ( state: post_create) :
    topic=state["topic"]
    messages=[ SystemMessage  (content = "You are social platform post generator"),
                HumanMessage  (content = f"""Generate a post in 200 words on topic {topic} for publishing in my twitter!
                                            post must be creative, it should contain new content, humorous, informative
                                            there should not  be any critisisam """)
            ]
    post=llm.invoke(messages)

    return {"post": post.content}

def evaluate_post(state:post_create):
  
    post=state["post"]
    messages=[ SystemMessage  (content = "You are social platform post critic and evaluator"),
                HumanMessage  (content = f"""evaluate the the post: {state["post"]}, check whether this post is humourous, 
                                            realiasti, informative, new contect  and genetate evaluation in fallowing format \n
                                            evaluation : "Approved", "Not approved" \n
                                            feedback: Here mention merita snd demerita of the post""")

    
            ]
    result=structured_model.invoke(messages)
   
    return {"evaluation": result.evaluation, "feedback":result.feedback}

def optimize_post(state:post_create) :
    #feedback = state["fedback"]
    #evaluation = state["evaluation"]
    iteration= state["iteration"] +1
    messages=[ SystemMessage  (content = "You are social platform post critic and evaluator and advisor"),
                HumanMessage  (content = f"""evaluate the the post based on feedback\n post: {state["post"]},
                                            feedback :{ state["fedback"]} \n on topic:{state["topic"]}  check whether this post is humourous, 
                                            realiasti, informative, new contect  and genetate new post for my twitter \n
                                            """)]
  
    post=llm.invoke(messages)

    return {"post": post.content, "iteration" : iteration }




def route_evaluation (state : post_create):
    evaluation=state["evaluation"]
    print(evaluation)
    iteration=state["iteration"]
    max_iteration=state["max_iteration"]

    if evaluation == "Approved" :
        return "Approved"

    else :
        return "Not approved"






In [173]:
graph = StateGraph(post_create)
graph.add_node("generate", generate)
graph.add_node("evaluate_post", evaluate_post)

graph.add_node("optimize_post", optimize_post)


graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate_post")
graph.add_conditional_edges("evaluate_post",route_evaluation,{"Approved":END, "Not approved":"optimize_post"})
graph.add_edge("optimize_post","evaluate_post")

wf=graph.compile()



In [ ]:
input= {"iteration" : 1, "topic": "Agentic AI", "max_iteration" : 5}
result=wf.invoke(input)




Approved
Here's a 200-word post on Agentic AI for Twitter:

"Imagine having a personal AI assistant that learns your habits and anticipates your needs! Welcome to the world of Agentic AI! This cutting-edge tech enables AI to take initiative, making decisions and acting on your behalf. 

Think of it like having a super-smart, ultra-organized virtual personal assistant. Need to schedule a meeting? Your Agentic AI has got it covered! Want to try a new restaurant? It'll already have made a reservation for you!

But that's not all - Agentic AI can also help you stay on top of your tasks, remind you of important deadlines, and even offer personalized recommendations. It's like having your own personal productivity coach!

The future of AI is all about empowerment, not replacement. With Agentic AI, you'll have more time to focus on what matters most - creating, innovating, and living your best life! #AgenticAI #AI #FutureOfTech"


In [179]:
print(result["post"])

Here's a 200-word post on Agentic AI for Twitter:

"Imagine having a personal AI assistant that learns your habits and anticipates your needs! Welcome to the world of Agentic AI! This cutting-edge tech enables AI to take initiative, making decisions and acting on your behalf. 

Think of it like having a super-smart, ultra-organized virtual personal assistant. Need to schedule a meeting? Your Agentic AI has got it covered! Want to try a new restaurant? It'll already have made a reservation for you!

But that's not all - Agentic AI can also help you stay on top of your tasks, remind you of important deadlines, and even offer personalized recommendations. It's like having your own personal productivity coach!

The future of AI is all about empowerment, not replacement. With Agentic AI, you'll have more time to focus on what matters most - creating, innovating, and living your best life! #AgenticAI #AI #FutureOfTech"
